# Predicting Airline Satisfaction (Kaggle Playground S6E10)

**Goal:** predict the probability that a passenger is satisfied with their flight.
**Metric:** ROC AUC, which only cares about how well the predictions *rank* satisfied passengers above unsatisfied ones.

**Final solution:** four gradient-boosting models trained on the same engineered features, combined by a stack (a logistic regression on their log-odds).

| Submission | Model | CV AUC | Public LB | Private LB |
|---|---|---|---|---|
| `03-blend-tuned-stack.csv` | Stack of LightGBM, XGBoost, HistGradientBoosting and CatBoost | **0.96110** | **0.96069** | **0.96138** |
| `04-lightgbm-optuna.csv` | LightGBM alone, the best single model | 0.96104 | 0.96063 | 0.96132 |

| Section | What it does |
|---|---|
| 1. Setup | imports, paths and the random seed |
| 2. Data | loads the tables and gives the columns short names |
| 3. Validation | the 5 fixed folds every model uses |
| 4. Features | preprocessing and the 46 + 7 model features |
| 5. Models | the 4 boosting models, cross-validated on the same folds |
| 6. Stack | combines the 4 models into one prediction |
| 7. Submission files | writes and checks the two files sent to Kaggle |

**Re-running:** each model saves its out-of-fold and test predictions in `preds/`, and a re-run loads them, so the whole notebook takes under a minute. Without saved predictions it retrains all 4 models: about 10 minutes on an 8-core laptop, with at most about 1.5 GB of RAM.

## 1. Setup

All settings live in the next cell: paths, the random seed, the target and id column names. Every later cell uses these names instead of typing paths or numbers again, so a change made here applies everywhere.

In [1]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder

# Paths and project-wide settings: change them here, never further down
DATA = Path("data")
PREDS = Path("preds")              # saved out-of-fold and test predictions
SUBMISSIONS = Path("submissions")
PREDS.mkdir(exist_ok=True)
SUBMISSIONS.mkdir(exist_ok=True)

SEED = 42
TARGET = "satisfaction"
ID_COL = "id"
N_FOLDS = 5
N_JOBS = -1                        # all CPU cores for every model

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 2. Data

There are three tables, plus the sample submission:

- `train.csv`: 699,635 passengers **with** the answer (`satisfaction`). The models learn from this table.
- `test.csv`: 299,844 passengers **without** the answer. We predict these rows and submit the predictions to Kaggle.
- `original.csv`: the real 129,880-row [Airline Satisfaction dataset](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k) that Kaggle used to generate the synthetic train and test data. The rules allow external data, and several features below are built from it.

In [2]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
original = pd.read_csv(DATA / "original.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

for name, df in [("train", train), ("test", test), ("original", original), ("sample", sample)]:
    print(f"{name:9s} {df.shape[0]:>8,} rows x {df.shape[1]} columns")

train      699,635 rows x 23 columns
test       299,844 rows x 22 columns
original   129,880 rows x 22 columns
sample     299,844 rows x 2 columns


The raw column names contain spaces and `/` (e.g. `Departure/Arrival time convenient`), so we rename everything to short snake_case names right after loading, and turn the target from `True/False` into `1/0`.

| Group | Columns | Meaning |
|---|---|---|
| Passenger | `gender`, `customer_type`, `age` | who is flying |
| Trip | `travel_type`, `cabin_class`, `distance` (miles) | what kind of trip |
| Service ratings | 13 columns, scored 0–5 (`wifi`, `online_boarding`, `seat_comfort`, …) | the survey answers; **0 = not rated** |
| Delays | `dep_delay`, `arr_delay` (minutes) | how late the flight left and arrived |
| Target | `satisfaction` | 1 = satisfied, 0 = neutral or dissatisfied |

In [3]:
RENAME = {
    "Age": "age",
    "Flight Distance": "distance",
    "Inflight wifi service": "wifi",
    "Departure/Arrival time convenient": "time_convenient",
    "Ease of Online booking": "online_booking",
    "Gate location": "gate_location",
    "Food and drink": "food_drink",
    "Online boarding": "online_boarding",
    "Seat comfort": "seat_comfort",
    "Inflight entertainment": "entertainment",
    "On-board service": "onboard_service",
    "Leg room service": "leg_room",
    "Baggage handling": "baggage",
    "Checkin service": "checkin",
    "Cleanliness": "cleanliness",
    "Departure Delay in Minutes": "dep_delay",
    "Arrival Delay in Minutes": "arr_delay",
    "Gender": "gender",
    "Customer Type": "customer_type",
    "Type of Travel": "travel_type",
    "Class": "cabin_class",
}
train = train.rename(columns=RENAME)
test = test.rename(columns=RENAME)
original = original.rename(columns=RENAME)

train[TARGET] = train[TARGET].astype(int)        # True/False -> 1/0
original[TARGET] = original[TARGET].astype(int)

NUMERIC = ["age", "distance", "dep_delay", "arr_delay"]
RATINGS = ["wifi", "time_convenient", "online_booking", "gate_location", "food_drink", "online_boarding",
           "seat_comfort", "entertainment", "onboard_service", "leg_room", "baggage", "checkin", "cleanliness"]
CATEGORICAL = ["gender", "customer_type", "travel_type", "cabin_class"]
RAW_FEATURES = NUMERIC + RATINGS + CATEGORICAL

print(f"{len(RAW_FEATURES)} raw features; satisfied passengers: {train[TARGET].mean():.1%} of train")

21 raw features; satisfied passengers: 44.4% of train


## 3. Validation

A model trained to tell train rows from test rows (adversarial validation) scores AUC 0.50: it can't tell them apart, so the test set is a random sample of the same synthetic data. That makes the choice simple: **stratified 5-fold cross-validation (CV).**

- We cut train into 5 parts, train on 4, predict the 5th, and repeat 5 times. "Stratified" means each part keeps the same 44.4% satisfied share.
- The folds are created **once** and reused by every model, so models can be compared fold by fold and their predictions stacked.
- The predictions each model makes for the rows it didn't train on are called **out-of-fold (OOF)** predictions. Their AUC over all 700k rows is our score.
- CV is trusted over the public leaderboard, which uses only ~60k test rows.

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(train, train[TARGET]))     # list of (train_idx, valid_idx) pairs

pd.DataFrame({
    "train rows": [len(tr) for tr, _ in folds],
    "valid rows": [len(va) for _, va in folds],
    "satisfied share (valid)": [round(train[TARGET].iloc[va].mean(), 4) for _, va in folds],
}, index=[f"fold {k}" for k in range(N_FOLDS)])

,train rows,valid rows,satisfied share (valid)
fold 0,559708,139927,0.4436
fold 1,559708,139927,0.4436
fold 2,559708,139927,0.4436
fold 3,559708,139927,0.4436
fold 4,559708,139927,0.4436


## 4. Features

Gradient-boosted trees need little preprocessing: no scaling (trees only compare values against thresholds), no one-hot encoding (the libraries read pandas `category` columns), and no outlier removal (long delays are real and rare). A rating of 0 stays a real value, because "not rated" carries signal.

`preprocess()` fills the few missing arrival delays, makes the category levels identical in every table, and shrinks the number types. It uses no target information, so it's safe to apply to train, test and original alike.

In [5]:
CATEGORY_LEVELS = {col: sorted(train[col].unique()) for col in CATEGORICAL}

def preprocess(df):
    out = df.copy()
    # 1) Missing arrival delays (~0.04%): copy the departure delay, which equals it in 87% of rows
    out["arr_delay"] = out["arr_delay"].fillna(out["dep_delay"])
    # 2) Text -> pandas 'category' with the SAME levels in every table, so category codes match
    for col in CATEGORICAL:
        out[col] = pd.Categorical(out[col], categories=CATEGORY_LEVELS[col])
    # 3) Smaller number types: 0-5 ratings fit in int8, distances in int16 (saves memory on 1M rows)
    out[RATINGS] = out[RATINGS].astype("int8")
    out["age"] = out["age"].astype("int8")
    out["distance"] = out["distance"].astype("int16")
    out[["dep_delay", "arr_delay"]] = out[["dep_delay", "arr_delay"]].astype("float32")
    if TARGET in out:
        out[TARGET] = out[TARGET].astype("int8")
    return out

train_p, test_p, original_p = preprocess(train), preprocess(test), preprocess(original)
y = train_p[TARGET].to_numpy()
print("Missing values left:", int(train_p[RAW_FEATURES].isna().sum().sum() + test_p[RAW_FEATURES].isna().sum().sum()
                                  + original_p[RAW_FEATURES].isna().sum().sum()))

Missing values left: 0


**The feature set.** On top of the 21 raw columns, the models get five groups of features. Each group improved CV AUC in all 5 folds; together they lift a LightGBM from 0.95862 (raw features) to 0.96041.

| Group | Columns | Idea | Uses the target? |
|---|---|---|---|
| Value counts | 7 | how often a distance, age, delay or pair occurs in train + test: the data generator left traces in these frequencies | no |
| Distance digits | 3 | the last three digits of the distance, another generator trace | no |
| Original lookups | 14 | satisfaction rate and passenger count per distance, age or segment in the original data | the original's labels only |
| Original model | 1 | the probability from a LightGBM trained on the original data alone | the original's labels only |
| Target encoding | 7 | satisfaction rate per distance, age and pairs in our train | **yes**, fitted inside each fold |

**Leakage rule:** a feature that uses train's target must never see the labels of the rows it is scored on, or the CV score becomes over-optimistic. Counts and digits don't use the target at all. The original lookups and model use only the original's labels, and validation never scores original rows. Target encoding uses train's labels, so it's fitted **inside each fold, on the training part only**.

### 4.1 Value counts

For each value (or pair of values), we count how many passengers in **train + test** have it. No target is used, so counting over test rows too is safe, and it gives more accurate counts.

In [6]:
COUNT_KEYS = [["distance"], ["age"], ["dep_delay"], ["arr_delay"],
              ["distance", "cabin_class"], ["distance", "travel_type"], ["age", "cabin_class"]]

def key_name(cols):
    return "_".join(cols)

def add_count_features(df, reference):
    out = df.copy()
    for cols in COUNT_KEYS:
        counts = reference.groupby(cols, observed=True).size().astype("int32").rename(f"count_{key_name(cols)}")
        out = out.merge(counts.reset_index(), on=cols, how="left")   # a left merge keeps the row order
    return out

reference = pd.concat([train_p[RAW_FEATURES], test_p[RAW_FEATURES]], ignore_index=True)
train_fe, test_fe = add_count_features(train_p, reference), add_count_features(test_p, reference)
COUNT_FEATURES = [f"count_{key_name(cols)}" for cols in COUNT_KEYS]

### 4.2 Distance digits

Synthetic data generators sometimes leave patterns in the digits of numbers: distances ending in 0, for example, are rarer in train (5.6%) than in the original (9.4%). A tree can't easily see "the last digit" in a raw number, so we hand it the digits as separate features. This exploits how the data was generated, not a real-world effect.

In [7]:
def add_digit_features(df):
    out = df.copy()
    out["distance_d1"] = (out["distance"] % 10).astype("int8")           # last digit
    out["distance_d2"] = (out["distance"] // 10 % 10).astype("int8")     # tens
    out["distance_d3"] = (out["distance"] // 100 % 10).astype("int8")    # hundreds
    return out

train_fe, test_fe = add_digit_features(train_fe), add_digit_features(test_fe)
DIGIT_FEATURES = ["distance_d1", "distance_d2", "distance_d3"]

### 4.3 Original-data lookups

For each key (e.g. a passenger's exact distance), we look up in the **original** data how many passengers had that value (`orig_n_…`) and what share of them were satisfied (`orig_rate_…`). Delays are left out because their distribution changed between the original and the synthetic data.

In [8]:
LOOKUP_KEYS = [["distance"], ["age"], ["distance", "cabin_class"], ["distance", "travel_type"],
               ["age", "cabin_class"], ["age", "travel_type"], ["customer_type", "travel_type", "cabin_class"]]

def add_original_lookups(df, original):
    out = df.copy()
    for cols in LOOKUP_KEYS:
        name = key_name(cols)
        stats = (original.groupby(cols, observed=True)[TARGET].agg(["mean", "size"])
                 .rename(columns={"mean": f"orig_rate_{name}", "size": f"orig_n_{name}"})
                 .astype("float32").reset_index())
        out = out.merge(stats, on=cols, how="left")   # values never seen in the original stay NaN
    return out

train_fe, test_fe = add_original_lookups(train_fe, original_p), add_original_lookups(test_fe, original_p)
LOOKUP_FEATURES = [f"orig_{kind}_{key_name(cols)}" for cols in LOOKUP_KEYS for kind in ("rate", "n")]

### 4.4 Original-data model

A compact way to use the original: **train a model on the original data alone** and give its predicted probability to our models as one extra feature. It never sees train's labels, so it's leakage-free. It is the models' strongest feature: they start from what the real data says about a passenger and correct it with the other columns.

In [9]:
original_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=SEED, verbose=-1)
original_model.fit(original_p[RAW_FEATURES], original_p[TARGET])

train_fe["orig_model_prob"] = original_model.predict_proba(train_fe[RAW_FEATURES])[:, 1].astype("float32")
test_fe["orig_model_prob"] = original_model.predict_proba(test_fe[RAW_FEATURES])[:, 1].astype("float32")
print(f"AUC of the original-only model on our train set: {roc_auc_score(y, train_fe['orig_model_prob']):.4f}")

AUC of the original-only model on our train set: 0.9527


### 4.5 Target encoding (inside the folds)

Target encoding replaces a value with **the satisfaction rate of passengers with that value in train**: `distance = 594` becomes the share of satisfied passengers among everyone in train who flew exactly 594 miles. It suits features with many values, like `distance` (3,474 values) and pairs like distance × class.

Inside each CV fold:
1. the encoder learns rates from the fold's **training part only**;
2. those rates are applied to the validation part and the test set;
3. for the training part itself, scikit-learn's `TargetEncoder` uses an internal 5-fold split, so no row's own label goes into its own encoding.

`gb_fold_inputs()` builds one fold's training, validation and test frames: the 46 model features plus the 7 encodings. Every model calls it, so all four see exactly the same inputs.

In [10]:
TE_KEYS = [["distance"], ["age"], ["distance", "cabin_class"], ["distance", "travel_type"],
           ["distance", "customer_type"], ["age", "cabin_class"], ["age", "travel_type"]]

def build_te_keys(df):
    # One integer per key; a pair (distance, class) becomes distance * 10 + class code
    keys = pd.DataFrame(index=df.index)
    for cols in TE_KEYS:
        key = df[cols[0]].astype("int32")
        for col in cols[1:]:
            key = key * 10 + df[col].cat.codes.astype("int32")
        keys[f"te_{key_name(cols)}"] = key
    return keys

def add_target_encoding(X_tr, y_tr, X_va, X_te, keys_tr, keys_va, keys_te):
    # Fit on the fold's training rows only; cross-fitting inside fit_transform protects the training rows themselves
    encoder = TargetEncoder(target_type="binary", cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    names = list(keys_tr.columns)
    parts = []
    for X, encoded in [(X_tr, encoder.fit_transform(keys_tr, y_tr)), (X_va, encoder.transform(keys_va)),
                       (X_te, encoder.transform(keys_te))]:
        parts.append(pd.concat([X.reset_index(drop=True),
                                pd.DataFrame(encoded.astype("float32"), columns=names)], axis=1))
    return parts

te_keys_train, te_keys_test = build_te_keys(train_fe), build_te_keys(test_fe)
TE_FEATURES = list(te_keys_train.columns)
MODEL_FEATURES = RAW_FEATURES + COUNT_FEATURES + DIGIT_FEATURES + LOOKUP_FEATURES + ["orig_model_prob"]


def gb_fold_inputs(tr_idx, va_idx):
    # Target encodings are fitted on this fold's training rows only, then applied to the valid and test rows
    return add_target_encoding(train_fe[MODEL_FEATURES].iloc[tr_idx], y[tr_idx],
                               train_fe[MODEL_FEATURES].iloc[va_idx], test_fe[MODEL_FEATURES],
                               te_keys_train.iloc[tr_idx], te_keys_train.iloc[va_idx], te_keys_test)


print(f"{len(MODEL_FEATURES)} model features + {len(TE_FEATURES)} target encodings")

46 model features + 7 target encodings


## 5. Models

Four gradient-boosting libraries. All of them build a long chain of small decision trees, where each new tree fixes the mistakes the chain still makes. They grow their trees differently, so they make slightly different mistakes, which is what the stack in section 6 needs:

| Model | How it grows trees | Settings |
|---|---|---|
| LightGBM | leaf by leaf: always splits the leaf with the biggest gain | tuned with Optuna (30 trials) |
| XGBoost | level by level: balanced trees | tuned with Optuna (30 trials) |
| HistGradientBoosting | leaf by leaf, scikit-learn's own booster, the least random | hand-picked |
| CatBoost | symmetric: the same split across a whole level | grid of 12 depth × L2 settings |

The searches ran on fold 0 at full size. The best settings were then retrained on all 5 folds and kept only if they beat the previous settings in at least 4 of the 5 folds. The settings below are the final ones.

Each model stops adding trees when its AUC on the validation fold stops improving (early stopping). That makes every score very slightly optimistic, by the same amount for all models.

`run_model_cv()` trains one model per fold, collects the out-of-fold predictions, averages the 5 test predictions, and saves both in `preds/`. If they're already saved, it loads them instead of retraining.

In [11]:
model_results = {}     # name -> scores, filled in by run_model_cv


def fold_aucs(pred):
    return np.array([roc_auc_score(y[va_idx], pred[va_idx]) for _, va_idx in folds])


def run_model_cv(name, fit_predict_fold):
    # fit_predict_fold(tr_idx, va_idx) trains on the fold's training rows and returns (valid predictions, test predictions)
    oof_path, test_path = PREDS / f"{name}-oof.npy", PREDS / f"{name}-test.npy"
    if oof_path.exists() and test_path.exists():
        oof, test_pred = np.load(oof_path), np.load(test_path)
        print(f"{name}: loaded saved predictions")
    else:
        oof, test_pred = np.zeros(len(y)), np.zeros(len(test_fe))
        start = time.time()
        for k, (tr_idx, va_idx) in enumerate(folds):
            fold_start = time.time()
            valid_pred, fold_test_pred = fit_predict_fold(tr_idx, va_idx)
            oof[va_idx] = valid_pred
            test_pred += np.asarray(fold_test_pred, dtype=float) / len(folds)
            print(f"  fold {k}: AUC {roc_auc_score(y[va_idx], valid_pred):.5f} ({time.time() - fold_start:.0f}s)")
        np.save(oof_path, oof)
        np.save(test_path, test_pred)
        print(f"{name}: trained in {time.time() - start:.0f}s")
    fold_auc = fold_aucs(oof)
    model_results[name] = {"oof_auc": roc_auc_score(y, oof), "fold_auc": fold_auc}
    print(f"{name}: OOF AUC {model_results[name]['oof_auc']:.5f}, folds {fold_auc.mean():.5f} ± {fold_auc.std():.5f}")
    return oof, test_pred

### 5.1 LightGBM

- **`colsample_bytree` 0.31:** each tree sees a random 31% of the columns. This was the search's biggest win. A likely reason: it stops every tree from leaning on the same few strong columns, such as `orig_model_prob`.
- **`num_leaves` 89, `min_child_samples` 13, `reg_lambda` 3.4:** fairly large trees, held back by an L2 penalty on the leaf values.
- **`learning_rate` 0.03** with early-stopping patience 300: small steps averaged over many trees. `n_estimators` is only a ceiling.

The tuned values are written with all their digits, so a retrain reproduces the saved predictions exactly.

In [12]:
LGB_PARAMS = dict(n_estimators=8000, learning_rate=0.03, num_leaves=89, min_child_samples=13,
                  reg_lambda=3.424280371272645, subsample=0.9815975557736427, subsample_freq=1,
                  colsample_bytree=0.3108428394413, random_state=SEED, verbose=-1, n_jobs=N_JOBS)


def lightgbm_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = gb_fold_inputs(tr_idx, va_idx)
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    # Early stopping on the validation fold's AUC; predictions then use the best iteration
    model.fit(X_tr, y[tr_idx], eval_X=X_va, eval_y=y[va_idx], eval_metric="auc",
              callbacks=[lgb.early_stopping(300, verbose=False)])
    print(f"  best iteration {model.best_iteration_:,}")
    preds = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    del model, X_tr, X_va, X_te
    gc.collect()
    return preds


oof_lgb, test_lgb = run_model_cv("lightgbm", lightgbm_fold)

lightgbm: loaded saved predictions


lightgbm: OOF AUC 0.96104, folds 0.96104 ± 0.00053


### 5.2 XGBoost

- **`max_depth` 9:** balanced trees with up to 512 leaves. They differ on purpose from LightGBM's lopsided trees.
- **`reg_lambda` 16, `min_child_weight` 0.63:** a strong L2 penalty on the leaf values, with almost no limit on leaf size.
- **`subsample` 0.86, `colsample_bytree` 0.47:** each tree sees 86% of the rows and 47% of the columns.
- **`learning_rate` 0.05** with patience 200. `enable_categorical=True` reads the pandas `category` columns directly.

In [13]:
XGB_PARAMS = dict(n_estimators=3500, learning_rate=0.05, max_depth=9, min_child_weight=0.6280146430083248,
                  reg_lambda=16.042151388937103, subsample=0.8597678370884065, colsample_bytree=0.47062067319518885,
                  tree_method="hist", enable_categorical=True, eval_metric="auc", early_stopping_rounds=200,
                  random_state=SEED, n_jobs=N_JOBS)


def xgboost_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = gb_fold_inputs(tr_idx, va_idx)
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y[tr_idx], eval_set=[(X_va, y[va_idx])], verbose=False)
    # best_iteration counts from 0, so the number of trees used is one more
    print(f"  best iteration {model.best_iteration + 1:,}")
    preds = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    del model, X_tr, X_va, X_te
    gc.collect()
    return preds


oof_xgb, test_xgb = run_model_cv("xgboost", xgboost_fold)

xgboost: loaded saved predictions


xgboost: OOF AUC 0.96082, folds 0.96083 ± 0.00058


### 5.3 HistGradientBoosting (scikit-learn)

- **`max_leaf_nodes` 63, `learning_rate` 0.05:** trees of up to 63 leaves, with moderate steps.
- **`max_features` 0.6:** each split looks at a random 60% of the features. The model has no row sampling, so this is its only source of randomness.
- **Early stopping on the validation fold** (`X_val`, `y_val`) with patience 50. It stops on the log loss, which is faster than computing the AUC after every tree. Unlike the other three, it keeps the extra trees it built while waiting, so a short patience is better here.

In [14]:
HGB_PARAMS = dict(max_iter=3000, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=40, l2_regularization=1.0,
                  max_features=0.6, categorical_features="from_dtype", early_stopping=True, scoring="loss",
                  n_iter_no_change=50, random_state=SEED)


def hist_gb_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = gb_fold_inputs(tr_idx, va_idx)
    model = HistGradientBoostingClassifier(**HGB_PARAMS)
    model.fit(X_tr, y[tr_idx], X_val=X_va, y_val=y[va_idx])
    print(f"  trees built {model.n_iter_:,}")
    preds = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    del model, X_tr, X_va, X_te
    gc.collect()
    return preds


oof_hgb, test_hgb = run_model_cv("hist-gb", hist_gb_fold)

hist-gb: loaded saved predictions


hist-gb: OOF AUC 0.96068, folds 0.96069 ± 0.00055


### 5.4 CatBoost

- **`depth` 7, `l2_leaf_reg` 3:** symmetric trees with exactly 2⁷ = 128 leaves, the best of the 12 grid settings.
- **`learning_rate` 0.08** with patience 150.
- **`one_hot_max_size` 4:** our categories have at most 3 values, so one-hot encoding them is ~2× faster than CatBoost's target statistics and just as accurate.
- **`boosting_type="Plain"`:** classic boosting. The "Ordered" mode protects small datasets against overfitting, but it is several times slower.

CatBoost reads categories as integers or strings, not as pandas `category`, so `catboost_codes()` turns them into their integer codes. Train and test share the same category levels, so the codes mean the same in both.

In [15]:
CATBOOST_PARAMS = dict(
    iterations=2500,            # upper limit; early stopping usually ends training sooner
    learning_rate=0.08,
    depth=7,                    # symmetric trees: 2**7 = 128 leaves
    l2_leaf_reg=3,
    one_hot_max_size=4,         # categories with <= 4 values are one-hot encoded (much faster than target statistics)
    boosting_type="Plain",      # "Ordered" is meant for small data and is several times slower
    eval_metric="AUC",
    early_stopping_rounds=150,
    random_seed=SEED,
    thread_count=N_JOBS,
    allow_writing_files=False,  # no catboost_info/ folder
    verbose=False,
)


def catboost_codes(X):
    X = X.copy()
    X[CATEGORICAL] = X[CATEGORICAL].apply(lambda col: col.cat.codes)
    return X


def catboost_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = (catboost_codes(X) for X in gb_fold_inputs(tr_idx, va_idx))
    model = CatBoostClassifier(cat_features=CATEGORICAL, **CATBOOST_PARAMS)
    model.fit(X_tr, y[tr_idx], eval_set=(X_va, y[va_idx]))   # keeps the trees up to the best validation AUC
    print(f"  best iteration {model.get_best_iteration() + 1:,}")
    preds = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    del model, X_tr, X_va, X_te
    gc.collect()
    return preds


oof_cat, test_cat = run_model_cv("catboost", catboost_fold)

catboost: loaded saved predictions


catboost: OOF AUC 0.96077, folds 0.96078 ± 0.00060


## 6. Stack

The stack combines the four models' predictions with one weight per model. A logistic regression learns the weights from the **out-of-fold** predictions, converted to log-odds and standardized. Each of those predictions was made by a model that never saw that passenger.

To check that the weights don't overfit, they're learned with **nested CV**: fitted on 4 folds, scored on the 5th, rotating through all 5. The final weights are then fitted on all the out-of-fold predictions and applied to the averaged test predictions.

In [16]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)          # only guards against log(0); keeps the ranking of very confident predictions
    return np.log(p / (1 - p))


def make_stacker():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


STACK_MODELS = {"LightGBM": (oof_lgb, test_lgb), "XGBoost": (oof_xgb, test_xgb),
                "HistGradientBoosting": (oof_hgb, test_hgb), "CatBoost": (oof_cat, test_cat)}
X_stack_oof = np.column_stack([logit(oof) for oof, _ in STACK_MODELS.values()])
X_stack_test = np.column_stack([logit(test_pred) for _, test_pred in STACK_MODELS.values()])

# Nested CV: every fold is scored with weights learned on the other 4 folds
stack_log_odds = np.zeros(len(y))
for tr_idx, va_idx in folds:
    stack_log_odds[va_idx] = make_stacker().fit(X_stack_oof[tr_idx], y[tr_idx]).decision_function(X_stack_oof[va_idx])
stack_oof = 1 / (1 + np.exp(-stack_log_odds))

# Final weights from all the out-of-fold predictions, applied to the test predictions
final_stack = make_stacker().fit(X_stack_oof, y)
stack_test = final_stack.predict_proba(X_stack_test)[:, 1]
np.save(PREDS / "stack-oof.npy", stack_oof)
np.save(PREDS / "stack-test.npy", stack_test)

weights = dict(zip(STACK_MODELS, final_stack[-1].coef_[0]))
summary = {label: {"CV AUC": roc_auc_score(y, oof), "stack weight": weights[label]}
           for label, (oof, _) in STACK_MODELS.items()}
summary["Stack"] = {"CV AUC": roc_auc_score(y, stack_oof)}
gain = fold_aucs(stack_oof) - fold_aucs(oof_lgb)
print(f"Stack vs LightGBM alone: {gain.mean():+.5f} per fold, better in {(gain > 0).sum()}/{N_FOLDS} folds")
pd.DataFrame(summary).T.style.format({"CV AUC": "{:.5f}", "stack weight": "{:.2f}"}, na_rep="")

Stack vs LightGBM alone: +0.00007 per fold, better in 5/5 folds


,CV AUC,stack weight
LightGBM,0.96104,1.41
XGBoost,0.96082,0.77
HistGradientBoosting,0.96068,0.29
CatBoost,0.96077,0.54
Stack,0.96110,


**What this shows:**
- **The stack scores CV AUC 0.96110,** above every single model, and beats the best one, LightGBM, in all 5 folds (+0.00007 per fold).
- **LightGBM gets the largest weight** (about 1.4), XGBoost the next (0.8), and HistGradientBoosting and CatBoost smaller ones. The weights are on standardized log-odds, so they are directly comparable.
- The four models are highly correlated, so the gain is small, but it comes from honest nested CV: every fold is scored with weights that never saw it.

## 7. Submission files

Kaggle wants one row per test passenger, `id,satisfaction`, with a probability. The two final submissions are chosen by CV, not by the public leaderboard:
- `03-blend-tuned-stack.csv`: the stack, the best CV (0.96110);
- `04-lightgbm-optuna.csv`: LightGBM alone (0.96104), a different and simpler candidate in case the stack disappoints.

Before a file is written, it's checked: the columns match `sample_submission.csv`; it has 299,844 rows with the sample's ids in the same order; there are no missing values; and every value is between 0 and 1. Creating the files doesn't submit them: uploading is a separate step (`kaggle competitions submit`).

In [17]:
def make_submission(file_name, test_pred):
    submission = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), TARGET: test_pred})
    # Checks before anything is written
    assert list(submission.columns) == list(sample.columns), "columns must match sample_submission.csv"
    assert len(submission) == len(sample) == 299_844, "one row per test passenger"
    assert (submission[ID_COL].to_numpy() == sample[ID_COL].to_numpy()).all(), "ids in the same order as the sample"
    assert submission[TARGET].notna().all(), "no missing predictions"
    assert submission[TARGET].between(0, 1).all(), "probabilities must be in [0, 1]"
    path = SUBMISSIONS / file_name
    submission.to_csv(path, index=False)
    print(f"{path}: {len(submission):,} rows, predictions {submission[TARGET].min():.4f} to {submission[TARGET].max():.4f}, "
          f"mean {submission[TARGET].mean():.4f}")
    return submission


stack_submission = make_submission("03-blend-tuned-stack.csv", stack_test)
lightgbm_submission = make_submission("04-lightgbm-optuna.csv", test_lgb)

submissions/03-blend-tuned-stack.csv: 299,844 rows, predictions 0.0077 to 0.9958, mean 0.4437


submissions/04-lightgbm-optuna.csv: 299,844 rows, predictions 0.0071 to 0.9961, mean 0.4437


Both files pass every check. On the leaderboard they ranked exactly as CV predicted: the stack scored 0.96069 public and **0.96138 private**, LightGBM alone 0.96063 public and 0.96132 private.